# Processing MESACLIP data - ocean daily, single ensembles

* **Description**: Reads in MESACLIP Historical Ocean data
* **Creator**: Alice DuVivier
* **Date**: October 2026

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import xesmf as xe
import matplotlib.pyplot as plt
import scipy.sparse as sps

import dask
from distributed import Client
from ncar_jobqueue import NCARCluster

## Spin up DASK cluster

In [ ]:
# Create our NCAR Cluster - which uses PBSCluster under the hood
cluster = NCARCluster(walltime='12:00:00', cores=1, processes=1, memory='100 GB', 
                      resource_spec='select=1:ncpus=1:mem=100GB',interface='ext')

# Spin up 32 workers
cluster.scale(32)

# Assign the cluster to our Client
client = Client(cluster)

In [ ]:
client
# cluster.close()

## Set MESACLIP ensemble information

In [ ]:
# Historical
path_base = '/gdex/data/d651007/'
case_base = 'b.e13.BHISTC5.ne120_t12.cesm-ihesp-'
# list 10 members of MESACLIP - last one is the first and named differently
ens_mems = ['sehires38-1850-2005.001', 'hires1.0.30-1920-2005.002', 
            'hires1.0.30-1920-2005.003', 'hires1.0.44-1920-2005.004', 
            'hires1.0.44-1920-2005.005', 'hires1.0.45-1920-2005.006', 
            'hires1.0.46-1920-2005.007', 'hires1.0.46-1920-2005.008',
            'hires1.0.46-1920-2005.009', 'hires1.0.46-1920-2005.010']

nens = len(ens_mems)

In [ ]:
# set the groups of years since we have to load them separately
# 1920s, 1930s, 1940s, 1950s, 1960s, 1970s, 1980s, 1990s, 2000s
year_sets = ['192','193','194','195','196',
             '197','198','199']

nyears = len(year_sets)

## Load in Sea Ice Data

In [ ]:
# Choose variable to load
var_in = 'SSH'

In [ ]:
# set output directory
path_out = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/daily/'+var_in+'/'

In [ ]:
# loop through each ensemble and each set of years

# loop through ensembles
for n in np.arange(0,nens,1)[1:10]:
    # loop through year groupings
    for y in np.arange(0,nyears,1):
        ens_in = ens_mems[n]
        yr_pre = year_sets[y]
        print('loading ensemble: '+ens_in+' for years '+yr_pre+'0s')
        # set case
        case = case_base + ens_in
        # set data directory
        path = path_base+case+'/ocn/proc/tseries/day_1'
        # get list of files
        files = sorted(glob(f'{path}/{case}.pop.h.nday1.{var_in}.{yr_pre}?????-????????.nc'))       
        # load files
        ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                           concat_dim="time", combine='nested', decode_times=True)
        # take an average over the time bounds to get the right time dimension
        ds_tmp["time"] = ds_tmp.time_bound.compute().mean(dim="d2")
        # crop to just the Alaskan Arctic Region: 50-80N Lat, 170-230 Lon
        #ds_sub = ds.where((ds.TLAT >= 50.0) & (ds.TLAT <= 80.0) & (ds.TLON >= 170.0) & (ds.TLON <= 230.0))
        ds_sub = ds_tmp.isel(nlat=slice(1700,2250)).isel(nlon=slice(2700,3300))

        # save and export data
        # set output file name
        nn = n+1
        if nn < 10:
            ensnum = '00'+str(nn)
        elif nn >= 10:
            ensnum = '010'

        file_out = 'MESACLIP_HIST_'+ensnum+'_'+yr_pre+'0-'+yr_pre+'9'+'_'+var_in+'_alaska_daily.nc'
        fout = path_out + file_out
        #print(fout)

        # make output array
        ds_out = ds_sub
        # change the attributes
        ds_out.attrs['author'] = 'Alice DuVivier'
        ds_out.attrs['date_processed'] = datetime.now().strftime('%Y-%m-%d')
        ds_out.attrs['contents'] = f'Subset of historical MESACLIP Data'
        # calculate the size of the dataset in GB
        size_gb = ds_out.nbytes / (1024**3)
        print(f"The dataset is approximately {size_gb:.2f} GB.")

        # Export the dataset to NetCDF with all attributes and coordinates
        print('Writing post processed file')
        ds_out.to_netcdf(fout)


In [ ]:
print("Done!")
cluster.close()